In [70]:
import pandas as pd
import numpy as np
import time
import math
import datetime
import seaborn as sns
import matplotlib.pylab as plt
import sklearn
import os

import dask.dataframe as dd # To work with large data sets
from sklearn.preprocessing import StandardScaler

In [71]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


In [72]:

if IN_COLAB:
    drive.mount('/content/drive')
else:
    print("Not running in Colab, skipping Drive mount.")

Not running in Colab, skipping Drive mount.


In [73]:
# Some globals

PROJECT_PATH = '.'
if IN_COLAB:
    PROJECT_PATH = '/content/drive/MyDrive/CENG574_PROJECT'

DATA_PATH = os.path.join(PROJECT_PATH, 'data')

available_datasets = ['cleaned_partitions', 'nyc_taxi_grid_sampled.parquet', 'proportional_sampled_2k.parquet']
DATASET_NAME = available_datasets[0] # <------------------  Here we pick which of them to be normalized 

FEATURE_SELECTION_DATA_PATH = os.path.join(DATA_PATH, DATASET_NAME) 
NORMALIZED_DATA_PATH = os.path.join(DATA_PATH, 'normalized_' + DATASET_NAME) # Our selected features (after normalization)


In [74]:

df =  pd.read_parquet(FEATURE_SELECTION_DATA_PATH)
print(f"[INFO] Loaded data set with total samples: {len(df)}")

[INFO] Loaded data set with total samples: 8716


In [75]:

df_dropna = df.dropna() # Only drops NaN, Inf remains

# stats:
print(f"Number of samples in the original dataset: {df.shape[0]}")
print(f"Number of samples with all valid measurements: {df_dropna.shape[0]}")
print(f"Number of samples with missing/invalid values overall: {df.shape[0] - df_dropna.shape[0]}")
print(f"Number of features: {df.shape[1]}")



Number of samples in the original dataset: 8716
Number of samples with all valid measurements: 8716
Number of samples with missing/invalid values overall: 0
Number of features: 16


In [76]:
# To inspect -Inf, Inf, NaN
# because otherwise normalization fails
bad_rows = df[
    df.isna().any(axis=1) |
    df.isin([np.inf, -np.inf]).any(axis=1)
]

print(bad_rows.head())
print("Number of bad rows (missing ):", len(bad_rows))

Empty DataFrame
Columns: [passenger_count, pickup_longitude, pickup_latitude, dropoff_longitude, dropoff_latitude, PULocationID, DOLocationID, Borough, Speed, hour_sin, hour_cos, day_sin, day_cos, log_trip_times, log_trip_distance, log_total_amount]
Index: []
Number of bad rows (missing ): 0


In [77]:
df = df.drop(bad_rows.index)
print("Number of samples after dropping inf and nans:", len(df))

Number of samples after dropping inf and nans: 8716


In [78]:
LABEL_COLUMNS = [
    "PULocationID",
    "DOLocationID",
    "Borough"
]

labels_df = df[LABEL_COLUMNS].copy()

features_df = df.drop(columns=LABEL_COLUMNS)

In [79]:
# Below is for sanity check, to make sure all of our features are numeric

def to_numpy_numeric(df):
    df = df.select_dtypes(include=[np.number])
    return df.to_numpy(), df.columns

X, feature_names = to_numpy_numeric(features_df)
print(X.shape)
print(feature_names)

(8716, 13)
Index(['passenger_count', 'pickup_longitude', 'pickup_latitude',
       'dropoff_longitude', 'dropoff_latitude', 'Speed', 'hour_sin',
       'hour_cos', 'day_sin', 'day_cos', 'log_trip_times', 'log_trip_distance',
       'log_total_amount'],
      dtype='object')


In [80]:
# Below is for sanity check, in our project no constant found for the selected features

def remove_constant_feats(dataset: np.ndarray, epsilon=1e-12) -> np.ndarray:
    # Check if there is 0 std in a feature,
    # this is done to avoid division by zero during nornmalization
    constant_data_idxs = dataset.std(axis=0) < epsilon

    if np.any(constant_data_idxs):
        idxs = np.where(constant_data_idxs == True)
        print("[WARNING] Found zero std in the following features at indices: ", idxs, " discarding these features...")
        dataset = dataset[:,~constant_data_idxs] # keep positive std
        print(f"[INFO] Dataset now has shape {dataset.shape}")

    else:
      print("[INFO] No constant (std = 0) feature found. Returning the same dataset...")

    return dataset

X = remove_constant_feats(X)

[INFO] No constant (std = 0) feature found. Returning the same dataset...


## Normalize Data Set


In [81]:
def assert_normalized(dataset: np.ndarray):
    assert np.allclose(dataset.std(axis=0), 1.0), f"Std check failed for: {dataset.std(axis=0)}"
    assert np.allclose(dataset.mean(axis=0), 0.0), f"Mean check failed for: {dataset.mean(axis=0)}"
    print(f"[INFO] Asserted that the dataset is normalized. Dataset mean {dataset.mean():.4f}, std {dataset.std():.4f}")

scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)
assert_normalized(X_normalized) # For sanity check

[INFO] Asserted that the dataset is normalized. Dataset mean 0.0000, std 1.0000


## Save


We normalized the features (all numeric, sanity checked)
and now we concatanate the ground truth labels back to the
dataset for exporting.

In [82]:

df_norm_features = pd.DataFrame(
    X_normalized,
    columns=feature_names,
    index=df.index
)

df_norm = pd.concat(
    [df_norm_features, labels_df],
    axis=1
)


In [83]:
import pyarrow as pa
import pyarrow.parquet as pq
import os
os.makedirs(NORMALIZED_DATA_PATH, exist_ok=True)


table = pa.Table.from_pandas(df_norm)

pq.write_to_dataset(
    table,
    root_path=NORMALIZED_DATA_PATH
)

print(f"Parquet with shape {df_norm.shape} saved to {NORMALIZED_DATA_PATH}")

Parquet with shape (8716, 16) saved to .\data\normalized_nyc_taxi_grid_sampled.parquet
